# Labour Productivity, Unit Labour Costs, and Rates: Australia vs United States

Five standalone charts comparing labour-market and rates dynamics in Australia and the United States.

1. **Whole-economy labour productivity (annual)** — OECD GDP per hour worked. Annual because no agency publishes a quarterly whole-economy GDP-per-hour series for the US.
2. **Labour productivity, market sector / non-farm business (quarterly)** — US `OPHNFB` from FRED vs Australian market-sector gross value added per hour worked (ABS 5206.0 series `A3606058X`).
3. **Unit labour costs (quarterly)** — Employment-based ULC from OECD productivity database (`ULCE`).
4. **10-year government bond yields (daily)** — Australian Government 10-year bond yield (RBA F2) vs US 10-year Treasury constant maturity (FRED `DGS10`).
5. **Central bank policy rates (monthly)** — RBA Official Cash Rate vs US Effective Federal Funds Rate (FRED `FEDFUNDS`).

Charts 1–3 are indexed to 2019 levels with a log-linear pre-pandemic trend overlay. Charts 4–5 show raw rates.


## Python setup

In [1]:
# system imports
from pathlib import Path

In [2]:
# analytic imports
import numpy as np
import pandas as pd
import requests
import re
import matplotlib.pyplot as plt

import readabs as ra
from readabs import metacol as mc
import mgplot as mg
from abs_prices import get_cpi


In [3]:
# pandas display
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999

## Constants

In [4]:
SHOW = False
FILE_TYPE = "png"

CHART_DIR = "./CHARTS/Productivity AU vs US/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

# Sources
SOURCE = "Sources: ABS 5206.0; FRED OPHNFB; OECD productivity database"
FOOTER_NOTE = (
    "Seasonally adjusted. Trend: log-linear fit on 2010–2019 (quarterly: 2010Q1–2019Q4), extrapolated."
)

# Index base periods
BASE_QUARTER = pd.Period("2019Q3", freq="Q")
BASE_YEAR = pd.Period("2019", freq="Y-DEC")

# Trend fit window
Q_FIT_START, Q_FIT_END = pd.Period("2010Q1", freq="Q"), pd.Period("2019Q4", freq="Q")
A_FIT_START, A_FIT_END = pd.Period("2010", freq="Y-DEC"), pd.Period("2019", freq="Y-DEC")

# Plot start (display only)
Q_PLOT_START = pd.Period("2010Q1", freq="Q")
A_PLOT_START = pd.Period("2010", freq="Y-DEC")

# Colours: AU = blue, US = darkorange, trends = grey
AU_COLOUR = "blue"
US_COLOUR = "darkorange"
TREND_COLOUR = "#888888"

FRED_API_KEY = Path("fred.api").read_text().strip()

## Helper functions

In [5]:
def _fetch_fred(series_id: str, frequency: str | None = None, start: str = "1990-01-01") -> dict:
    """Low-level FRED observations fetch. Returns the raw observations list."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start,
    }
    if frequency is not None:
        params["frequency"] = frequency
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()["observations"]


def fetch_fred_quarterly(series_id: str) -> pd.Series:
    """Fetch a quarterly FRED series indexed by Period[Q]."""
    obs = _fetch_fred(series_id, frequency="q")
    return (
        pd.Series(
            {pd.Period(o["date"], freq="Q"): float(o["value"]) for o in obs if o["value"] != "."}
        )
        .sort_index()
        .rename(series_id)
    )


def fetch_fred_monthly(series_id: str) -> pd.Series:
    """Fetch a monthly FRED series indexed by Period[M]."""
    obs = _fetch_fred(series_id, frequency="m")
    return (
        pd.Series(
            {pd.Period(o["date"], freq="M"): float(o["value"]) for o in obs if o["value"] != "."}
        )
        .sort_index()
        .rename(series_id)
    )


def fetch_fred_daily(series_id: str) -> pd.Series:
    """Fetch a daily FRED series indexed by Period[D]."""
    obs = _fetch_fred(series_id, frequency="d")
    return (
        pd.Series(
            {pd.Period(o["date"], freq="D"): float(o["value"]) for o in obs if o["value"] != "."}
        )
        .sort_index()
        .rename(series_id)
    )


In [6]:
def fetch_oecd(dataset: str, series_code: str, freq: str) -> pd.Series:
    """Fetch one series from an OECD dataset via DB.nomics.

    Args:
        dataset: e.g. "DSD_PDB@DF_PDB" (annual productivity) or "DSD_PDB@DF_PDB_ULC_Q" (quarterly ULC).
        series_code: full DB.nomics series code.
        freq: pandas PeriodIndex freq, e.g. "Y-DEC" or "Q".
    """
    url = f"https://api.db.nomics.world/v22/series/OECD/{dataset}/{series_code}?observations=1"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    s = response.json()["series"]["docs"][0]
    return (
        pd.Series(s["value"], index=pd.PeriodIndex(s["period"], freq=freq))
        .dropna()
        .rename(series_code)
    )

In [7]:
def rebase(series: pd.Series, base_period: pd.Period) -> pd.Series:
    """Rebase a series to 100 at the supplied base period."""
    return series / series.loc[base_period] * 100.0


def fit_log_linear_trend(
    series: pd.Series,
    fit_start: pd.Period,
    fit_end: pd.Period,
    extrap_end: pd.Period,
) -> pd.Series:
    """Fit log-linear trend on [fit_start, fit_end] and extrapolate to extrap_end.

    Returns a Series spanning [fit_start, extrap_end] at the series' native frequency.
    """
    sub = series.loc[fit_start:fit_end]
    x_fit = np.arange(len(sub))
    log_y = np.log(sub.values)
    slope, intercept = np.polyfit(x_fit, log_y, 1)
    full_index = pd.period_range(fit_start, extrap_end, freq=series.index.freq)
    x_full = np.arange(len(full_index))
    return pd.Series(np.exp(intercept + slope * x_full), index=full_index)

In [8]:
def plot_chart(
    *,
    au_actual: pd.Series,
    us_actual: pd.Series,
    au_trend: pd.Series,
    us_trend: pd.Series,
    plot_start: pd.Period,
    title: str,
    ylabel: str,
    source: str,
    footer: str,
) -> None:
    """Plot one standalone chart: AU + US actuals plus pre-pandemic trends.

    Uses mgplot.line_plot_finalise, which handles save + close.
    The chart is saved into the current mgplot chart directory using ``title`` as the filename.
    """
    df = pd.DataFrame(
        {
            "US trend (pre-COVID)": us_trend.loc[plot_start:],
            "Australia trend (pre-COVID)": au_trend.loc[plot_start:],
            "United States": us_actual.loc[plot_start:],
            "Australia": au_actual.loc[plot_start:],
        }
    )
    mg.line_plot_finalise(
        df,
        color=[TREND_COLOUR, TREND_COLOUR, US_COLOUR, AU_COLOUR],
        width=[1.0, 1.0, 2.0, 2.0],
        style=["--", ":", "-", "-"],
        annotate=[False, False, True, True],
        rounding=1,
        fontsize="small",
        dropna=True,
        title=title,
        ylabel=ylabel,
        rfooter=source,
        lfooter=footer,
        legend={"loc": "upper left", "fontsize": "x-small", "frameon": False},
        axhline={"y": 100, "color": "black", "linewidth": 0.5, "zorder": -1},
        show=SHOW,
        file_type=FILE_TYPE,
    )

## Data: ABS quarterly market-sector productivity

In [9]:
abs_table = "5206001_Key_Aggregates"
abs_data, abs_meta = ra.read_abs_cat("5206.0", single_excel_only=abs_table, verbose=False)

_, id_mkt, _ = ra.find_abs_id(
    abs_meta,
    {
        abs_table: mc.table,
        "Gross value added per hour worked market sector: Index ;": mc.did,
        "Seasonally Adjusted": mc.stype,
    },
    verbose=False,
)
au_market_q = abs_data[abs_table][id_mkt].dropna()
assert not au_market_q.empty, "ABS market-sector productivity series came back empty"
au_market_q.tail()

Series ID
2025Q1     99.8
2025Q2    100.5
2025Q3    100.7
2025Q4    100.9
2026Q1    100.2
Freq: Q-DEC, Name: A3606058X, dtype: float64

## Data: FRED US non-farm business productivity

In [10]:
us_nfb_q = fetch_fred_quarterly("OPHNFB")
assert not us_nfb_q.empty, "FRED OPHNFB came back empty"
us_nfb_q.tail()

2025Q1    116.187
2025Q2    117.385
2025Q3    118.884
2025Q4    119.350
2026Q1    119.437
Freq: Q-DEC, Name: OPHNFB, dtype: float64

## Data: OECD annual GDP per hour worked

In [11]:
# OECD productivity database (annual): GDP per hour worked, chain-linked volume (rebased),
# non-transformed annual data in national currency per hour.
us_gdp_a = fetch_oecd("DSD_PDB@DF_PDB", "USA.A.GDPHRS._T.XDC_H.LR.N._Z._Z", "Y-DEC")
au_gdp_a = fetch_oecd("DSD_PDB@DF_PDB", "AUS.A.GDPHRS._T.XDC_H.LR.N._Z._Z", "Y-DEC")
assert not us_gdp_a.empty and not au_gdp_a.empty, "OECD GDP/hour series came back empty"
pd.concat({"US": us_gdp_a.tail(), "AU": au_gdp_a.tail()}, axis=1)

,US,AU
2021,81.520205,101.423933
2022,81.101429,97.685897
2023,82.268856,97.758096
2024,84.114714,97.169940
2025,85.602364,NaN
2020,NaN,99.885277


## Data: OECD quarterly unit labour costs

In [12]:
# OECD productivity database (quarterly): unit labour costs, employment-based, index, current prices,
# seasonally adjusted, not calendar adjusted.
us_ulc_q = fetch_oecd("DSD_PDB@DF_PDB_ULC_Q", "USA.Q.ULCE._T.IX.V._Z.S.NC", "Q")
au_ulc_q = fetch_oecd("DSD_PDB@DF_PDB_ULC_Q", "AUS.Q.ULCE._T.IX.V._Z.S.NC", "Q")
assert not us_ulc_q.empty and not au_ulc_q.empty, "OECD ULC series came back empty"
pd.concat({"US": us_ulc_q.tail(), "AU": au_ulc_q.tail()}, axis=1)

,US,AU
2024Q3,124.4998,NaN
2024Q4,125.3433,133.7473
2025Q1,127.2331,135.3950
2025Q2,126.8633,135.8363
2025Q3,127.2089,137.8219
2025Q4,NaN,138.3967


In [13]:
# Quarterly series: rebase to 2019-Q3 = 100
us_nfb_q_idx = rebase(us_nfb_q, BASE_QUARTER)
au_market_q_idx = rebase(au_market_q, BASE_QUARTER)
us_ulc_q_idx = rebase(us_ulc_q, BASE_QUARTER)
au_ulc_q_idx = rebase(au_ulc_q, BASE_QUARTER)

# Annual series: rebase to 2019 = 100
us_gdp_a_idx = rebase(us_gdp_a, BASE_YEAR)
au_gdp_a_idx = rebase(au_gdp_a, BASE_YEAR)

In [14]:
# Trend extrapolation runs to the latest available actual observation in each pairing
q_prod_end = max(us_nfb_q_idx.index[-1], au_market_q_idx.index[-1])
q_ulc_end = max(us_ulc_q_idx.index[-1], au_ulc_q_idx.index[-1])
a_end = max(us_gdp_a_idx.index[-1], au_gdp_a_idx.index[-1])

us_nfb_q_trend = fit_log_linear_trend(us_nfb_q_idx, Q_FIT_START, Q_FIT_END, q_prod_end)
au_market_q_trend = fit_log_linear_trend(au_market_q_idx, Q_FIT_START, Q_FIT_END, q_prod_end)
us_ulc_q_trend = fit_log_linear_trend(us_ulc_q_idx, Q_FIT_START, Q_FIT_END, q_ulc_end)
au_ulc_q_trend = fit_log_linear_trend(au_ulc_q_idx, Q_FIT_START, Q_FIT_END, q_ulc_end)
us_gdp_a_trend = fit_log_linear_trend(us_gdp_a_idx, A_FIT_START, A_FIT_END, a_end)
au_gdp_a_trend = fit_log_linear_trend(au_gdp_a_idx, A_FIT_START, A_FIT_END, a_end)

summary = pd.DataFrame(
    {
        "latest": [
            us_gdp_a_idx.iloc[-1], au_gdp_a_idx.iloc[-1],
            us_nfb_q_idx.iloc[-1], au_market_q_idx.iloc[-1],
            us_ulc_q_idx.iloc[-1], au_ulc_q_idx.iloc[-1],
        ],
        "trend": [
            us_gdp_a_trend.loc[us_gdp_a_idx.index[-1]],
            au_gdp_a_trend.loc[au_gdp_a_idx.index[-1]],
            us_nfb_q_trend.loc[us_nfb_q_idx.index[-1]],
            au_market_q_trend.loc[au_market_q_idx.index[-1]],
            us_ulc_q_trend.loc[us_ulc_q_idx.index[-1]],
            au_ulc_q_trend.loc[au_ulc_q_idx.index[-1]],
        ],
        "period": [
            us_gdp_a_idx.index[-1], au_gdp_a_idx.index[-1],
            us_nfb_q_idx.index[-1], au_market_q_idx.index[-1],
            us_ulc_q_idx.index[-1], au_ulc_q_idx.index[-1],
        ],
    },
    index=[
        "US whole-economy GDP/hr (annual)",
        "AU whole-economy GDP/hr (annual)",
        "US non-farm business productivity",
        "AU market sector productivity",
        "US unit labour costs",
        "AU unit labour costs",
    ],
)
summary["gap_pct"] = (summary["latest"] / summary["trend"] - 1) * 100
summary

,latest,trend,period,gap_pct
US whole-economy GDP/hr (annual),111.792974,104.011693,2025,7.481160
AU whole-economy GDP/hr (annual),98.945660,106.858066,2024,-7.404594
US non-farm business productivity,114.700996,105.344174,2026Q1,8.882144
AU market sector productivity,102.663934,113.782787,2026Q1,-9.771999
US unit labour costs,119.871449,110.514227,2025Q3,8.466984
AU unit labour costs,129.571944,104.986194,2025Q4,23.418079


## Plots

In [15]:
plot_chart(
    au_actual=au_gdp_a_idx,
    us_actual=us_gdp_a_idx,
    au_trend=au_gdp_a_trend,
    us_trend=us_gdp_a_trend,
    plot_start=A_PLOT_START,
    title="Whole-economy labour productivity (annual): Australia vs United States",
    ylabel="Index, 2019 = 100",
    source="Source: OECD productivity database",
    footer="GDP per hour worked. Trend: log-linear fit on 2010–2019, extrapolated.",
)

In [16]:
plot_chart(
    au_actual=au_market_q_idx,
    us_actual=us_nfb_q_idx,
    au_trend=au_market_q_trend,
    us_trend=us_nfb_q_trend,
    plot_start=Q_PLOT_START,
    title="Labour productivity (quarterly): Australia vs United States",
    ylabel="Index, 2019-Q3 = 100",
    source="Sources: ABS 5206.0; FRED OPHNFB",
    footer="AU: market-sector GVA/hr. US: non-farm business output/hr. Both SA. Trend: log-linear, 2010Q1–2019Q4.",
)

In [17]:
plot_chart(
    au_actual=au_ulc_q_idx,
    us_actual=us_ulc_q_idx,
    au_trend=au_ulc_q_trend,
    us_trend=us_ulc_q_trend,
    plot_start=Q_PLOT_START,
    title="Unit labour costs (quarterly): Australia vs United States",
    ylabel="Index, 2019-Q3 = 100",
    source="Source: OECD productivity database",
    footer="Employment-based ULC, current prices, SA. Trend: log-linear fit on 2010Q1–2019Q4, extrapolated.",
)

## Plot: Side-by-side productivity (whole-economy annual + market sector quarterly)

In [18]:
def two_digit_years(label: str) -> str:
    """Shorten 4-digit years to 2 digits (e.g. 2010 -> 10)."""
    return re.sub(r"\b(?:19|20)(\d{2})\b", r"\1", label)


def plot_productivity_side_by_side() -> None:
    """Two-panel chart: annual whole-economy (left) + quarterly market sector (right)."""
    _fig, (ax_a, ax_q) = plt.subplots(1, 2, figsize=(9, 4.5))

    # Left panel: whole-economy (annual)
    annual_df = pd.DataFrame(
        {
            "US trend (pre-COVID)": us_gdp_a_trend.loc[A_PLOT_START:],
            "Australia trend (pre-COVID)": au_gdp_a_trend.loc[A_PLOT_START:],
            "United States": us_gdp_a_idx.loc[A_PLOT_START:],
            "Australia": au_gdp_a_idx.loc[A_PLOT_START:],
        }
    )
    mg.line_plot(
        annual_df,
        ax=ax_a,
        color=[TREND_COLOUR, TREND_COLOUR, US_COLOUR, AU_COLOUR],
        width=[1.0, 1.0, 2.0, 2.0],
        style=["--", ":", "-", "-"],
        annotate=[False, False, True, True],
        rounding=1,
        fontsize="small",
        dropna=True,
        tick_relabel=two_digit_years,
    )
    mg.finalise_plot(
        ax_a,
        axes_only=True,
        title="Whole-economy (annual)",
        ylabel="Index, 2019 = 100",
        legend={"loc": "upper left", "fontsize": "x-small", "frameon": False},
        axhline={"y": 100, "color": "black", "linewidth": 0.5, "zorder": -1},
    )

    # Right panel: market sector / non-farm business (quarterly)
    quarterly_df = pd.DataFrame(
        {
            "US trend (pre-COVID)": us_nfb_q_trend.loc[Q_PLOT_START:],
            "Australia trend (pre-COVID)": au_market_q_trend.loc[Q_PLOT_START:],
            "United States": us_nfb_q_idx.loc[Q_PLOT_START:],
            "Australia": au_market_q_idx.loc[Q_PLOT_START:],
        }
    )
    mg.line_plot(
        quarterly_df,
        ax=ax_q,
        color=[TREND_COLOUR, TREND_COLOUR, US_COLOUR, AU_COLOUR],
        width=[1.0, 1.0, 2.0, 2.0],
        style=["--", ":", "-", "-"],
        annotate=[False, False, True, True],
        rounding=1,
        fontsize="small",
        dropna=True,
        tick_relabel=two_digit_years,
    )

    # Closing finalise: panel styling plus figure-level title, footers, save and close
    mg.finalise_plot(
        ax_q,
        title="Market sector (quarterly)",
        ylabel="Index, 2019-Q3 = 100",
        legend={"loc": "upper left", "fontsize": "x-small", "frameon": False},
        axhline={"y": 100, "color": "black", "linewidth": 0.5, "zorder": -1},
        suptitle="Labour productivity: Australia vs United States",
        lfooter="Australia. Seasonally adjusted. Trend: log-linear fit on pre-pandemic data, extrapolated.",
        rfooter="Sources: OECD; ABS 5206.0; FRED OPHNFB",
        figsize=(9, 4.5),
        dpi=300,
        file_type=FILE_TYPE,
        show=SHOW,
    )


plot_productivity_side_by_side()

Mismatched type: 'figsize=(9, 4.5)' must be of type 'tuple[float, float]', in
finalise_plot().


## Data: 10-year government bond yields (daily)


In [19]:
# AU 10-year via RBA F2 (daily, from 2013-05). US 10-year via FRED DGS10 (daily, from 1990).
rba_f2_data, _ = ra.read_rba_table("F2", verbose=False)
au_10y_d = rba_f2_data["FCMYGBAG10D"].dropna().astype(float).rename("Australia")
us_10y_d = fetch_fred_daily("DGS10").rename("United States")
yields_df = pd.concat([us_10y_d, au_10y_d], axis=1).sort_index()
assert not yields_df.empty, "No yield observations"
yields_df.tail()

,United States,Australia
2026-05-27,4.48,4.863
2026-05-28,4.45,NaN
2026-05-29,4.45,NaN
2026-06-01,4.47,NaN
2026-06-02,4.46,NaN


## Plot: 10-year government bond yields

In [20]:
au_last_val, au_last_dt = au_10y_d.iloc[-1], au_10y_d.index[-1]
us_last_val, us_last_dt = us_10y_d.iloc[-1], us_10y_d.index[-1]
latest = (
    f"Latest:  AU {au_last_val:.2f}% ({au_last_dt.strftime('%d-%b-%Y')})"
    f"   US {us_last_val:.2f}% ({us_last_dt.strftime('%d-%b-%Y')})"
)

mg.line_plot_finalise(
    yields_df.loc[max(au_10y_d.first_valid_index(), us_10y_d.first_valid_index()):],
    color=[US_COLOUR, AU_COLOUR],
    width=[1.25, 1.25],
    style=["-", "-"],
    annotate=True,
    rounding=2,
    fontsize="small",
    dropna=True,
    title="10-year government bond yields (daily): Australia vs United States",
    ylabel="Per cent per annum",
    rheader=latest,
    rfooter="Sources: RBA F2; FRED DGS10",
    lfooter="AU: 10-year Australian Government bond yield (interpolated). US: 10-year Treasury constant maturity.",
    legend={"loc": "upper left", "fontsize": "x-small", "frameon": False},
    show=SHOW,
    file_type=FILE_TYPE,
)

## Data: Central bank policy rates (monthly)


In [21]:
# AU: RBA Official Cash Rate (monthly via readabs). US: Effective Federal Funds Rate (FRED FEDFUNDS).
au_ocr_m = ra.read_rba_ocr(monthly=True, verbose=False).rename("Australia")
us_ff_m = fetch_fred_monthly("FEDFUNDS").rename("United States")
policy_df = pd.concat([us_ff_m, au_ocr_m], axis=1).sort_index()
assert not policy_df.empty, "No policy rate observations"
policy_df.tail()

,United States,Australia
2026-02,3.64,3.85
2026-03,3.64,4.10
2026-04,3.64,4.10
2026-05,3.63,4.35
2026-06,NaN,4.35


## Plot: Central bank policy rates

In [22]:
au_pr_val, au_pr_dt = au_ocr_m.iloc[-1], au_ocr_m.index[-1]
us_pr_val, us_pr_dt = us_ff_m.iloc[-1], us_ff_m.index[-1]
policy_latest = (
    f"Latest:  AU {au_pr_val:.2f}% ({au_pr_dt.strftime('%b-%Y')})"
    f"   US {us_pr_val:.2f}% ({us_pr_dt.strftime('%b-%Y')})"
)

mg.line_plot_finalise(
    policy_df.loc[max(au_ocr_m.first_valid_index(), us_ff_m.first_valid_index()):],
    color=[US_COLOUR, AU_COLOUR],
    width=[1.5, 1.5],
    style=["-", "-"],
    drawstyle="steps-post",
    annotate=True,
    rounding=2,
    fontsize="small",
    dropna=True,
    title="Central bank policy rates (monthly): Australia vs United States",
    ylabel="Per cent",
    rheader=policy_latest,
    rfooter="Sources: RBA A2; FRED FEDFUNDS",
    lfooter="AU: RBA OCR target (steps on RBA decisions). US: effective FFR, monthly avg (smoothed across change months).",
    legend={"loc": "upper right", "fontsize": "x-small", "frameon": False},
    show=SHOW,
    file_type=FILE_TYPE,
)

In [23]:
%load_ext watermark
%watermark -u -n -t -v -iv -w

Last updated: Fri, 05 Jun 2026 09:44:26

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.0

matplotlib: 3.10.9
mgplot    : 0.2.25
numpy     : 2.4.6
pandas    : 3.0.3
pathlib   : 1.0.1
re        : 2.2.1
readabs   : 0.2.2
requests  : 2.34.2

Watermark: 2.6.0



## Data: Real industrial electricity prices (quarterly)

AU: ABS 6427.0 Table 6427013 series `A2309192C` — PPI input price of
*Electricity* to manufacturing (quarterly index, 1968-Q3+).

US: EIA `PRICE.US-IND.M` via DBnomics — monthly average retail price of
electricity to industrial customers (cents/kWh, 2001-01+), resampled to quarterly mean.

Both series are deflated by the respective country's CPI All Groups and rebased to
2010-Q1 = 100, so the chart is a real-terms comparison.

In [24]:
def fetch_dbnomics_series(provider: str, dataset: str, code: str, freq: str) -> pd.Series:
    """Generic DBnomics series fetch (used here for EIA electricity prices)."""
    url = (
        f'https://api.db.nomics.world/v22/series/{provider}/{dataset}/{code}'
        f'?observations=1'
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    s = response.json()['series']['docs'][0]
    return (
        pd.Series(s['value'], index=pd.PeriodIndex(s['period'], freq=freq))
        .dropna()
        .rename(code)
    )

In [25]:
# AU: PPI input price of electricity to manufacturing
ppi_data, _ = ra.read_abs_cat('6427.0', single_excel_only='6427013', verbose=False)
au_elec_ppi_q = ppi_data['6427013']['A2309192C'].dropna()

# AU CPI All groups SA
au_cpi_q = get_cpi("headline_sa")[0]

# US: EIA monthly cents/kWh industrial -> quarterly average
us_elec_m = fetch_dbnomics_series('EIA', 'ELEC', 'PRICE.US-IND.M', 'M')
us_elec_q = us_elec_m.groupby(us_elec_m.index.asfreq('Q-DEC')).mean()

# US CPI All Urban Consumers
us_cpi_q = fetch_fred_quarterly('CPIAUCSL')

In [26]:
# Real industrial electricity indices, rebased to 2010-Q1 = 100
ELEC_BASE = pd.Period('2010Q1', freq='Q')

au_elec_real = rebase((au_elec_ppi_q / au_cpi_q).dropna(), ELEC_BASE)
us_elec_real = rebase((us_elec_q / us_cpi_q).dropna(), ELEC_BASE)

pd.concat(
    {'AU real index': au_elec_real, 'US real index': us_elec_real}, axis=1,
).loc[ELEC_BASE:].iloc[::8].round(1)

,AU real index,US real index
2010Q1,100.0,100.0
2012Q1,113.2,94.1
2014Q1,140.2,99.4
2016Q1,121.5,90.2
2018Q1,151.2,90.6
2020Q1,158.9,82.4
2022Q1,130.4,85.1
2024Q1,123.9,83.9
2026Q1,118.9,90.7


## Plot: Real industrial electricity prices

In [27]:
elec_df = pd.DataFrame({
    'United States': us_elec_real,
    'Australia':     au_elec_real,
}).loc[ELEC_BASE:]

au_last = au_elec_real.iloc[-1]
us_last = us_elec_real.iloc[-1]
elec_latest = (
    f'Latest (real, 2010-Q1 = 100):  '
    f'AU {au_last:.0f} ({au_elec_real.index[-1]})   '
    f'US {us_last:.0f} ({us_elec_real.index[-1]})'
)

mg.line_plot_finalise(
    elec_df,
    color=[US_COLOUR, AU_COLOUR],
    width=[2.0, 2.0],
    annotate=True,
    rounding=0,
    fontsize='small',
    dropna=True,
    title='Real industrial electricity prices (quarterly): Australia vs United States',
    ylabel='Index, 2010-Q1 = 100, real (CPI-deflated)',
    rheader=elec_latest,
    rfooter='Sources: ABS 6427.0, 6401.0; EIA; FRED',
    lfooter='AU: PPI for electricity inputs to manufacturing. US: EIA industrial retail.',
    legend={'loc': 'upper left', 'fontsize': 'x-small', 'frameon': False},
    axhline={'y': 100, 'color': 'black', 'linewidth': 0.5, 'zorder': -1},
    show=SHOW,
    file_type=FILE_TYPE,
)

## Data: Manufacturing share of GDP (quarterly)

AU: Manufacturing GVA / GDP, current prices, seasonally adjusted, from
ABS 5206.0 — series `A85231878A` (Manufacturing GVA, table 5206045) and
`A2304418T` (GDP, table 5206001). Quarterly current-price industry GVA in
5206 starts in 2002-Q3, which is the binding history constraint.

US: BEA Value Added by Industry: Manufacturing as a Percentage of GDP,
via FRED `VAPGDPMA` (quarterly).

In [28]:
# AU: Manufacturing GVA (current prices, SA) / GDP (current prices, SA), quarterly
gva_data, _ = ra.read_abs_cat(
    '5206.0', single_excel_only='5206045_Industry_GVA_Current_Price', verbose=False,
)
au_mfg_gva_q = gva_data['5206045_Industry_GVA_Current_Price']['A85231878A'].dropna()

ka_data, _ = ra.read_abs_cat(
    '5206.0', single_excel_only='5206001_Key_Aggregates', verbose=False,
)
au_gdp_cur_q = ka_data['5206001_Key_Aggregates']['A2304418T'].dropna()

au_mfg_share_q = (au_mfg_gva_q / au_gdp_cur_q * 100).dropna().rename('Australia')

# US: BEA Mfg as % of GDP via FRED, quarterly
us_mfg_share_q = fetch_fred_quarterly('VAPGDPMA').rename('United States')

mfg_df = pd.concat([us_mfg_share_q, au_mfg_share_q], axis=1).sort_index()
mfg_df.tail()

,United States,Australia
2025Q1,9.4,5.411361
2025Q2,9.4,5.219746
2025Q3,9.5,5.111096
2025Q4,9.4,5.078863
2026Q1,NaN,5.154487


In [29]:
mg.line_plot_finalise(
    mfg_df.loc[Q_PLOT_START:],
    color=[US_COLOUR, AU_COLOUR],
    width=[1.5, 1.5],
    annotate=True,
    rounding=1,
    fontsize='small',
    dropna=True,
    title='Manufacturing share of GDP (quarterly): Australia vs United States',
    ylabel='Per cent of GDP',
    rfooter='Sources: ABS 5206.0; FRED VAPGDPMA (BEA)',
    lfooter='AU: Mfg GVA / GDP, current prices, SA. US: BEA Value Added by Industry / GDP.',
    legend={'loc': 'upper right', 'fontsize': 'x-small', 'frameon': False},
    show=SHOW,
    file_type=FILE_TYPE,
)